In [ ]:
import json
import numpy as np
import pandas as pd
from collections import Counter

# ── RUTAS ─────────────────────────────────────────────────────────────────────
VAL_DATA_PATH  = "../data/last_task/val_new.json"       # tu JSON de validación
PREDS_PATH     = "../data/last_task/xlm-roberta-base-80.6.json"   # el JSON que guarda evaluate()

# ── CARGA ─────────────────────────────────────────────────────────────────────
with open(VAL_DATA_PATH,  "r") as f: val_data  = json.load(f)
with open(PREDS_PATH,     "r") as f: preds_raw = json.load(f)

preds = preds_raw["predictions"]
assert len(preds) == len(val_data), f"Mismatch: {len(preds)} preds vs {len(val_data)} samples"

# ── CONSTRUCCIÓN DEL DATAFRAME ────────────────────────────────────────────────
rows = []
for sample, pred in zip(val_data, preds):
    votes     = sample["label"] if isinstance(sample["label"], list) else [sample["label"]]
    n_votes   = len(votes)
    n_pos     = sum(votes)
    certainty = abs(n_pos / n_votes - 0.5) * 2   # 0 = total desacuerdo, 1 = acuerdo total

    has_ocr   = bool(sample.get("qwen_ocr",          "").strip())
    has_trans = bool(sample.get("qwen_transcription", "").strip())
    has_reas  = bool(sample.get("qwen_reasoning",     "").strip())

    physio     = sample.get("physio", {})
    n_eeg      = len(physio.get("EEG", []))
    n_et       = len(physio.get("ET",  []))
    n_hr       = len(physio.get("HR",  []))

    rows.append({
        "id":          sample["id_Tiktok"],
        "label":       pred["label"],
        "pred":        pred["pred"],
        "prob":        pred["prob"],              # prob de clase 1
        "correct":     pred["label"] == pred["pred"],
        "certainty":   round(certainty, 3),
        "n_annotators": n_votes,
        "has_ocr":     has_ocr,
        "has_trans":   has_trans,
        "has_reas":    has_reas,
        "n_modalities": int(has_ocr) + int(has_trans) + int(has_reas),
        "n_eeg":       n_eeg,
        "n_et":        n_et,
        "n_hr":        n_hr,
        "has_physio":  n_eeg > 0 or n_et > 0,
        "ocr_len":     len((sample.get("qwen_ocr")          or "").split()),
        "trans_len":   len((sample.get("qwen_transcription") or "").split()),
        "reas_len":    len((sample.get("qwen_reasoning")     or "").split()),
    })

df = pd.DataFrame(rows)

# ─────────────────────────────────────────────────────────────────────────────
# 1. RESUMEN GLOBAL
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix, f1_score

print("=" * 60)
print("1. RESUMEN GLOBAL")
print("=" * 60)
print(classification_report(df["label"], df["pred"], target_names=["No sarcasmo", "Sarcasmo"]))

cm = confusion_matrix(df["label"], df["pred"])
cm_df = pd.DataFrame(cm,
    index   = ["Real: No sarcasmo", "Real: Sarcasmo"],
    columns = ["Pred: No sarcasmo", "Pred: Sarcasmo"]
)
print("\nMatriz de confusión:")
print(cm_df.to_string())

# ─────────────────────────────────────────────────────────────────────────────
# 2. ERRORES POR CERTEZA DEL ANNOTATOR
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("2. ACCURACY POR CERTEZA DEL ANNOTATOR")
print("=" * 60)

df["certainty_bin"] = pd.cut(df["certainty"], bins=[0, 0.25, 0.5, 0.75, 1.01],
                              labels=["0-25% (muy ambiguo)", "25-50%", "50-75%", "75-100% (claro)"])
cert_table = df.groupby("certainty_bin", observed=True).agg(
    n          = ("correct", "count"),
    accuracy   = ("correct", "mean"),
    f1_macro   = ("correct", lambda x: f1_score(
                    df.loc[x.index, "label"],
                    df.loc[x.index, "pred"], average="macro", zero_division=0)),
    prob_media = ("prob",    "mean"),
).round(3)
print(cert_table.to_string())

# ─────────────────────────────────────────────────────────────────────────────
# 3. ERRORES POR MODALIDAD TEXTUAL DISPONIBLE
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("3. ACCURACY POR MODALIDADES TEXTO DISPONIBLES")
print("=" * 60)

for col, name in [("has_ocr", "OCR"), ("has_trans", "Transcripción"), ("has_reas", "Reasoning")]:
    tbl = df.groupby(col).agg(
        n        = ("correct", "count"),
        accuracy = ("correct", "mean"),
    ).round(3)
    tbl.index = tbl.index.map({
        False: f"Sin {name}",
        True:  f"Con {name}"
    })
    print(f"\n{name}:")
    print(tbl.to_string())

print("\nPor nº de modalidades texto disponibles:")
print(df.groupby("n_modalities").agg(
    n        = ("correct", "count"),
    accuracy = ("correct", "mean"),
).round(3).to_string())

# ─────────────────────────────────────────────────────────────────────────────
# 4. ERRORES POR FISIOLOGÍA
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("4. ACCURACY CON/SIN FISIOLOGÍA")
print("=" * 60)

print(df.groupby("has_physio").agg(
    n        = ("correct", "count"),
    accuracy = ("correct", "mean"),
    f1_macro = ("correct", lambda x: f1_score(
                    df.loc[x.index, "label"],
                    df.loc[x.index, "pred"], average="macro", zero_division=0)),
).round(3).rename(index={True: "Con fisiología", False: "Sin fisiología"}).to_string())

# ─────────────────────────────────────────────────────────────────────────────
# 5. DISTRIBUCIÓN DE PROBABILIDADES — DONDE EL MODELO DUDA
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("5. DISTRIBUCIÓN DE PROBABILIDADES")
print("=" * 60)

df["prob_bin"] = pd.cut(df["prob"], bins=[0, 0.3, 0.4, 0.5, 0.6, 0.7, 1.0],
                         labels=["0-30", "30-40", "40-50", "50-60", "60-70", "70-100"])
prob_table = df.groupby("prob_bin", observed=True).agg(
    n           = ("correct", "count"),
    n_correct   = ("correct", "sum"),
    accuracy    = ("correct", "mean"),
).round(3)
prob_table["n_correct"] = prob_table["n_correct"].astype(int)
print(prob_table.to_string())

uncertain = df[(df["prob"] > 0.4) & (df["prob"] < 0.6)]
print(f"\nMuestras en zona de duda (prob 0.4-0.6): {len(uncertain)} "
      f"({100*len(uncertain)/len(df):.1f}% del total)")
print(f"Accuracy en zona de duda: {uncertain['correct'].mean():.3f}")

# ─────────────────────────────────────────────────────────────────────────────
# 6. PEORES PREDICCIONES — CASOS CONCRETOS PARA INSPECCIONAR
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("6. FALSOS POSITIVOS MÁS CONFIADOS (predijo sarcasmo, era no sarcasmo)")
print("=" * 60)

fp = df[(df["pred"] == 1) & (df["label"] == 0)].sort_values("prob", ascending=False).head(10)
print(fp[["id", "prob", "certainty", "has_ocr", "has_trans", "has_reas", "n_eeg"]].to_string(index=False))

print("\nFALSOS NEGATIVOS MÁS CONFIADOS (predijo no sarcasmo, era sarcasmo)")
fn = df[(df["pred"] == 0) & (df["label"] == 1)].sort_values("prob", ascending=True).head(10)
print(fn[["id", "prob", "certainty", "has_ocr", "has_trans", "has_reas", "n_eeg"]].to_string(index=False))

# ─────────────────────────────────────────────────────────────────────────────
# 7. LONGITUD DE TEXTO VS ERROR
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("7. LONGITUD DE TEXTO VS ACCURACY")
print("=" * 60)

for col, name in [("ocr_len", "OCR"), ("trans_len", "Transcripción"), ("reas_len", "Reasoning")]:
    df[f"{col}_bin"] = pd.qcut(df[col], q=3, labels=["corto", "medio", "largo"], duplicates="drop")
    tbl = df.groupby(f"{col}_bin", observed=True).agg(
        n        = ("correct", "count"),
        accuracy = ("correct", "mean"),
    ).round(3)
    print(f"\n{name} por longitud:")
    print(tbl.to_string())

# ─────────────────────────────────────────────────────────────────────────────
# 8. RESUMEN FINAL — QUÉ ESTÁ FALLANDO
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("8. RESUMEN: PATRONES DE ERROR")
print("=" * 60)

error_df = df[~df["correct"]]
print(f"Total errores: {len(error_df)} / {len(df)} ({100*len(error_df)/len(df):.1f}%)\n")

print("Distribución de errores por certeza del annotator:")
print(error_df["certainty_bin"].value_counts().sort_index().to_string())

print("\nDistribución de errores por nº modalidades texto:")
print(error_df["n_modalities"].value_counts().sort_index().to_string())

print("\nErrores con vs sin fisiología:")
print(error_df["has_physio"].value_counts().rename({True: "Con fisiología", False: "Sin fisiología"}).to_string())

print("\nThreshold óptimo para F1 macro:")
best_t, best_f1 = 0.5, 0.0
for t in np.arange(0.2, 0.8, 0.01):
    preds_t = (df["prob"] >= t).astype(int)
    f1      = f1_score(df["label"], preds_t, average="macro")
    if f1 > best_f1:
        best_f1, best_t = f1, t
print(f"  Threshold={best_t:.2f}  →  F1 macro={best_f1:.4f}  (actual con 0.5: "
      f"{f1_score(df['label'], df['pred'], average='macro'):.4f})")

# ─────────────────────────────────────────────────────────────────────────────
# 9. ANÁLISIS DETALLADO: CASOS EXTREMOS (CERTEZA 100% Y 0%)
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("9. CASOS EXTREMOS: ERRORES CLAROS Y MÁXIMA AMBIGÜEDAD")
print("=" * 60)

# --- 9.1 ERRORES EN CASOS CLARÍSIMOS (Certeza 100%) ---
# Humanos están totalmente de acuerdo, pero el modelo falla.
errores_100_cert = df[(df["certainty"] == 1.0) & (~df["correct"])]

print(f"\n[!] Errores en casos CLAROS (Certeza 100%): {len(errores_100_cert)} muestras")

if not errores_100_cert.empty:
    # Separar en Falsos Positivos y Falsos Negativos
    fp_claros = errores_100_cert[errores_100_cert["pred"] == 1].sort_values("prob", ascending=False)
    fn_claros = errores_100_cert[errores_100_cert["pred"] == 0].sort_values("prob", ascending=True)

    print(f"\n  -> Falsos Positivos (Era NO sarcasmo indiscutible, pero predijo SARCASMO): {len(fp_claros)}")
    if not fp_claros.empty:
        print(fp_claros[["id", "prob", "n_modalities", "has_physio"]].head(10).to_string(index=False))

    print(f"\n  -> Falsos Negativos (Era SARCASMO indiscutible, pero predijo NO sarcasmo): {len(fn_claros)}")
    if not fn_claros.empty:
        print(fn_claros[["id", "prob", "n_modalities", "has_physio"]].head(10).to_string(index=False))

# --- 9.2 CASOS DE MÁXIMA AMBIGÜEDAD (Certeza 0%) ---
# Humanos están divididos (ej. 1 a favor, 1 en contra). 
casos_0_cert = df[df["certainty"] == 0.0]

print(f"\n[?] Casos de máxima AMBIGÜEDAD (Certeza 0%): {len(casos_0_cert)} muestras")

if not casos_0_cert.empty:
    print(f"  -> Accuracy del modelo en estos casos (prácticamente al azar): {casos_0_cert['correct'].mean():.3f}")
    
    # Ver casos ambiguos donde el modelo está "demasiado seguro" de sí mismo
    # Probabilidad > 0.8 (muy seguro de sarcasmo) o < 0.2 (muy seguro de no sarcasmo)
    seguros_en_ambiguos = casos_0_cert[(casos_0_cert["prob"] >= 0.8) | (casos_0_cert["prob"] <= 0.2)]
    
    print(f"\n  -> Casos donde los humanos dudan al máximo, pero el modelo está MUY SEGURO (prob >= 0.8 o <= 0.2): {len(seguros_en_ambiguos)}")
    if not seguros_en_ambiguos.empty:
        # Ordenamos por la 'distancia' al centro para ver los más extremos primero
        seguros_en_ambiguos = seguros_en_ambiguos.assign(extremo=abs(seguros_en_ambiguos["prob"] - 0.5))
        seguros_en_ambiguos = seguros_en_ambiguos.sort_values("extremo", ascending=False)
        print(seguros_en_ambiguos[["id", "label", "pred", "prob"]].head(10).to_string(index=False))

# ─────────────────────────────────────────────────────────────────────────────
# 10. SESGO DE GÉNERO Y ANÁLISIS DE TRUNCAMIENTO (ROBERTA LIMITS)
# ─────────────────────────────────────────────────────────────────────────────

# Expandimos la lista a inglés y términos comunes en estos datasets
gender_terms = [
    'woman', 'women', 'girl', 'her', 'she', 'ex', 'girlfriend', 'wife', 
    'mujer', 'mujeres', 'chica', 'ella', 'novia', 'esposa',
    'untrustworthy', 'lying', 'always', 'toxic', 'generalization'
]

def analyze_bias_and_length(row, tokenizer_estimate=1.3):
    # 1. Detección de términos
    reasoning = str(row.get('qwen_reasoning', '')).lower()
    has_gender = any(word in reasoning for word in gender_terms)
    
    # 2. Estimación de Tokens (XLM-R usa SentencePiece, aprox 1.3 tokens por palabra)
    # Si sumas todas las modalidades que le pasas al modelo:
    total_words = row['ocr_len'] + row['trans_len'] + row['reas_len']
    est_tokens = total_words * tokenizer_estimate
    
    return pd.Series([has_gender, est_tokens, est_tokens > 256])

df[['mention_gender', 'est_tokens', 'is_truncated']] = df.apply(analyze_bias_and_length, axis=1)

print("\n" + "=" * 60)
print("10. ANÁLISIS DE SESGO Y TRUNCAMIENTO (Límite 512 Tokens)")
print("=" * 60)

# --- Sub-análisis A: Género ---
print("\n[A] Rendimiento en casos con temática de Género/Relaciones:")
gender_stats = df.groupby('mention_gender').agg(
    n=('correct', 'count'),
    accuracy=('correct', 'mean'),
    avg_prob=('prob', 'mean')
).round(3)
print(gender_stats.rename(index={True: "Menciona Género", False: "Otros"}).to_string())

# --- Sub-análisis B: Truncamiento ---
print("\n[B] Rendimiento vs. Truncamiento (¿Se corta el texto?):")
trunc_stats = df.groupby('is_truncated').agg(
    n=('correct', 'count'),
    accuracy=('correct', 'mean'),
    avg_tokens=('est_tokens', 'mean')
).round(3)
print(trunc_stats.rename(index={True: "POSIBLE TRUNCADO (>512)", False: "Texto Completo"}).to_string())

# --- Sub-análisis C: El "Peor Escenario" ---
# Casos de género que además son largos (donde el modelo pierde el hilo)
bad_combo = df[(df['mention_gender'] == True) & (df['is_truncated'] == True)]
if not bad_combo.empty:
    print(f"\n[!] Casos críticos (Género + Truncados): {len(bad_combo)}")
    print(f"    Accuracy en este grupo: {bad_combo['correct'].mean():.3f}")

In [ ]:
"""
analyze_clear_cases.py
======================
Análisis focalizado en los casos con certeza de anotador = 100%
(todos los anotadores coinciden) para entender qué está fallando.

Ejecutar:
    python analyze_clear_cases.py
"""

import json
import textwrap
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, f1_score

# ── RUTAS ──────────────────────────────────────────────────────────────────────
VAL_DATA_PATH = "../data/last_task/val_new.json"
PREDS_PATH    = "../data/last_task/xlm-roberta-base-80.6.json"

# ── CARGA ──────────────────────────────────────────────────────────────────────
with open(VAL_DATA_PATH, "r") as f:
    val_data = json.load(f)
with open(PREDS_PATH, "r") as f:
    preds_raw = json.load(f)

preds = preds_raw["predictions"]
assert len(preds) == len(val_data), f"Mismatch: {len(preds)} preds vs {len(val_data)} samples"

# ── CONSTRUCCIÓN DEL DATAFRAME ─────────────────────────────────────────────────
rows = []
for sample, pred in zip(val_data, preds):
    votes     = sample["label"] if isinstance(sample["label"], list) else [sample["label"]]
    n_votes   = len(votes)
    n_pos     = sum(votes)
    certainty = abs(n_pos / n_votes - 0.5) * 2

    ocr   = (sample.get("qwen_ocr")          or "").strip()
    trans = (sample.get("qwen_transcription") or "").strip()
    reas  = (sample.get("qwen_reasoning")     or "").strip()

    rows.append({
        "id":           sample["id_Tiktok"],
        "label":        pred["label"],
        "pred":         pred["pred"],
        "prob":         pred["prob"],
        "correct":      pred["label"] == pred["pred"],
        "certainty":    round(certainty, 3),
        "n_annotators": n_votes,
        "n_pos_votes":  n_pos,
        "ocr":          ocr,
        "trans":        trans,
        "reas":         reas,
        "has_ocr":      bool(ocr),
        "has_trans":    bool(trans),
        "has_reas":     bool(reas),
        "ocr_len":      len(ocr.split()),
        "trans_len":    len(trans.split()),
        "reas_len":     len(reas.split()),
    })

df = pd.DataFrame(rows)

# ── HELPERS ────────────────────────────────────────────────────────────────────
SEP  = "=" * 70
sep2 = "-" * 70
W    = 90   # ancho para wrap de texto

def wrap(text, width=W, indent="    "):
    if not text:
        return f"{indent}[vacío]"
    lines = textwrap.wrap(text, width=width - len(indent))
    return "\n".join(f"{indent}{l}" for l in lines) if lines else f"{indent}[vacío]"

def print_sample(row, idx, total, tag=""):
    tipo = "✅ CORRECTO" if row["correct"] else "❌ ERROR"
    label_str = "Sarcasmo" if row["label"] == 1 else "No sarcasmo"
    pred_str  = "Sarcasmo" if row["pred"]  == 1 else "No sarcasmo"
    print(f"\n[{idx+1}/{total}] {tag}  {tipo}")
    print(f"  ID          : {row['id']}")
    print(f"  Real        : {label_str}  |  Predicción: {pred_str}  |  Prob(sarcasmo): {row['prob']:.3f}")
    print(f"  Anotadores  : {row['n_annotators']} votos, {row['n_pos_votes']} a favor de sarcasmo")
    print(f"  Certeza     : {row['certainty']:.3f}")
    print(f"  Modalidades : OCR={row['has_ocr']} ({row['ocr_len']} pal.) | "
          f"Trans={row['has_trans']} ({row['trans_len']} pal.) | "
          f"Reas={row['has_reas']} ({row['reas_len']} pal.)")

    if row["ocr"]:
        print(f"  --- OCR ---")
        print(wrap(row["ocr"]))
    if row["trans"]:
        print(f"  --- Transcripción ---")
        print(wrap(row["trans"]))
    if row["reas"]:
        print(f"  --- Reasoning ---")
        print(wrap(row["reas"]))
    print(sep2)


# ══════════════════════════════════════════════════════════════════════════════
# BLOQUE 1: VISIÓN GLOBAL DE LOS CASOS CON CERTEZA 100%
# ══════════════════════════════════════════════════════════════════════════════
clear = df[df["certainty"] == 1.0].copy()

print(SEP)
print("CASOS CON CERTEZA DE ANOTADOR = 100%")
print(SEP)
print(f"  Total casos claros : {len(clear)} / {len(df)} ({100*len(clear)/len(df):.1f}%)")
print(f"  Accuracy en claros : {clear['correct'].mean():.3f}")
print(f"  (Accuracy global)  : {df['correct'].mean():.3f}")
print()

print("  Distribución de etiquetas reales (claros):")
print(clear["label"].value_counts().rename({0: "No sarcasmo", 1: "Sarcasmo"}).to_string())
print()
print("  Classification report (solo casos claros):")
print(classification_report(clear["label"], clear["pred"],
                             target_names=["No sarcasmo", "Sarcasmo"], digits=3))

# F1 por subgrupo: errores vs aciertos
errors_clear   = clear[~clear["correct"]]
correct_clear  = clear[ clear["correct"]]

print(f"  Errores en casos claros   : {len(errors_clear)} ({100*len(errors_clear)/len(clear):.1f}%)")
print(f"  Aciertos en casos claros  : {len(correct_clear)} ({100*len(correct_clear)/len(clear):.1f}%)")

# Modalidades en errores vs aciertos
for col, name in [("has_ocr", "OCR"), ("has_trans", "Trans"), ("has_reas", "Reas")]:
    e_pct = errors_clear[col].mean()
    c_pct = correct_clear[col].mean()
    print(f"  {name:8s}: errores={e_pct:.2%}  aciertos={c_pct:.2%}")

# ══════════════════════════════════════════════════════════════════════════════
# BLOQUE 2: FALSOS POSITIVOS — "No sarcasmo pero predijo Sarcasmo"
# ══════════════════════════════════════════════════════════════════════════════
fp = clear[(clear["pred"] == 1) & (clear["label"] == 0)].sort_values("prob", ascending=False).reset_index(drop=True)

print()
print(SEP)
print(f"FALSOS POSITIVOS CLAROS: predijo SARCASMO, era NO SARCASMO  [{len(fp)} casos]")
print(SEP)

for i, row in fp.iterrows():
    print_sample(row, i, len(fp), tag="FP")

# ══════════════════════════════════════════════════════════════════════════════
# BLOQUE 3: FALSOS NEGATIVOS — "Sarcasmo pero predijo No sarcasmo"
# ══════════════════════════════════════════════════════════════════════════════
fn = clear[(clear["pred"] == 0) & (clear["label"] == 1)].sort_values("prob", ascending=True).reset_index(drop=True)

print()
print(SEP)
print(f"FALSOS NEGATIVOS CLAROS: predijo NO SARCASMO, era SARCASMO  [{len(fn)} casos]")
print(SEP)

for i, row in fn.iterrows():
    print_sample(row, i, len(fn), tag="FN")

# ══════════════════════════════════════════════════════════════════════════════
# BLOQUE 4: ACIERTOS EN CASOS CLAROS (referencia para comparar)
# ══════════════════════════════════════════════════════════════════════════════
print()
print(SEP)
print(f"ACIERTOS EN CASOS CLAROS (muestra aleatoria de 10)  [{len(correct_clear)} en total]")
print(SEP)

sample_correct = correct_clear.sample(min(10, len(correct_clear)), random_state=42).reset_index(drop=True)
for i, row in sample_correct.iterrows():
    print_sample(row, i, len(sample_correct), tag="OK")

# ══════════════════════════════════════════════════════════════════════════════
# BLOQUE 5: RESUMEN ESTADÍSTICO COMPARATIVO
# ══════════════════════════════════════════════════════════════════════════════
print()
print(SEP)
print("RESUMEN COMPARATIVO: ERRORES vs ACIERTOS (casos claros)")
print(SEP)

compare = pd.DataFrame({
    "Errores": {
        "n":            len(errors_clear),
        "prob_media":   errors_clear["prob"].mean(),
        "ocr_len_med":  errors_clear["ocr_len"].median(),
        "trans_len_med":errors_clear["trans_len"].median(),
        "reas_len_med": errors_clear["reas_len"].median(),
        "pct_has_reas": errors_clear["has_reas"].mean(),
        "pct_has_trans":errors_clear["has_trans"].mean(),
    },
    "Aciertos": {
        "n":            len(correct_clear),
        "prob_media":   correct_clear["prob"].mean(),
        "ocr_len_med":  correct_clear["ocr_len"].median(),
        "trans_len_med":correct_clear["trans_len"].median(),
        "reas_len_med": correct_clear["reas_len"].median(),
        "pct_has_reas": correct_clear["has_reas"].mean(),
        "pct_has_trans":correct_clear["has_trans"].mean(),
    }
}).round(3)

print(compare.to_string())

# ══════════════════════════════════════════════════════════════════════════════
# BLOQUE 6: THRESHOLD ÓPTIMO SOLO EN CASOS CLAROS
# ══════════════════════════════════════════════════════════════════════════════
print()
print(SEP)
print("THRESHOLD ÓPTIMO PARA F1 MACRO (solo casos claros)")
print(SEP)

best_t, best_f1 = 0.5, 0.0
for t in np.arange(0.2, 0.8, 0.01):
    preds_t = (clear["prob"] >= t).astype(int)
    f1 = f1_score(clear["label"], preds_t, average="macro", zero_division=0)
    if f1 > best_f1:
        best_f1, best_t = f1, t

current_f1 = f1_score(clear["label"], clear["pred"], average="macro", zero_division=0)
print(f"  Threshold actual (0.50) → F1 macro = {current_f1:.4f}")
print(f"  Threshold óptimo ({best_t:.2f}) → F1 macro = {best_f1:.4f}")
print(f"  Mejora potencial: +{best_f1 - current_f1:.4f}")

In [ ]:
"""
bayes_search.py
===============
Búsqueda bayesiana PARALELA de hiperparámetros para TF-IDF + XGBoost.
- Vocabulario y min_df calculados dinámicamente del corpus
- Stop words en inglés (el reasoning de Qwen, la modalidad más larga, está en inglés)
- Fix de serialización JSON para tipos numpy
- Paralelismo en CV y en XGBoost interno

Requisitos:
    pip install scikit-optimize xgboost scikit-learn joblib pandas numpy

Salida: lexical_preds_bayes.json
"""

import json
import warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.feature_extraction import text as sklearn_text
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold
from joblib import Parallel, delayed, cpu_count
from skopt import gp_minimize
from skopt.space import Real, Integer, Categorical
from skopt.utils import use_named_args
import xgboost as xgb

warnings.filterwarnings("ignore")

# ── RUTAS ──────────────────────────────────────────────────────────────────────
TRAIN_PATH  = "../data/last_task/train_new.json"
VAL_PATH    = "../data/last_task/val_new.json"
XLMR_PREDS  = "../data/last_task/xlm-roberta-base-79.6.json"
OUTPUT_PATH = "../data/last_task/lexical_preds_bayes.json"

N_CALLS     = 50
N_INITIAL   = 10
N_CV_FOLDS  = 5
N_JOBS_CV   = 6
N_JOBS_XGB  = 1
RANDOM_SEED = 42

N_CORES = cpu_count()
print(f"CPUs disponibles: {N_CORES}")

# ── HELPERS ────────────────────────────────────────────────────────────────────
def majority_label(label_field) -> int:
    votes = label_field if isinstance(label_field, list) else [label_field]
    return int(sum(votes) > len(votes) / 2)

def build_text(sample: dict) -> str:
    ocr   = (sample.get("qwen_ocr")          or "").strip()
    trans = (sample.get("qwen_transcription") or "").strip()
    reas  = (sample.get("qwen_reasoning")     or "").strip()
    if trans == "<no-speech>":
        trans = ""
    parts = []
    if ocr:   parts.append("OCR: " + ocr)
    if trans: parts.append("TRANS: " + trans)
    if reas:  parts.append("REAS: " + reas)
    return " ".join(parts)

# ── JSON ENCODER para tipos numpy ──────────────────────────────────────────────
class NpEncoder(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, np.bool_):    return bool(o)
        if isinstance(o, np.integer):  return int(o)
        if isinstance(o, np.floating): return float(o)
        if isinstance(o, np.ndarray):  return o.tolist()
        return super().default(o)

# ── CARGA ──────────────────────────────────────────────────────────────────────
with open(TRAIN_PATH, "r") as f: train_data = json.load(f)
with open(VAL_PATH,   "r") as f: val_data   = json.load(f)
with open(XLMR_PREDS, "r") as f: xlmr_raw   = json.load(f)

xlmr_preds     = xlmr_raw["predictions"]
xlmr_probs_val = np.array([p["prob"] for p in xlmr_preds])

assert len(xlmr_preds) == len(val_data), "Mismatch XLM-R preds vs val_data"

X_train = [build_text(s) for s in train_data]
y_train = np.array([majority_label(s["label"]) for s in train_data])

X_val   = [build_text(s) for s in val_data]
y_val   = np.array([majority_label(s["label"]) for s in val_data])

print(f"Train: {len(X_train)} samples | Val: {len(X_val)} samples")
print(f"Train pos/neg: {y_train.sum()} / {(1-y_train).sum()}")

scale_pos = float((1 - y_train).sum()) / max(y_train.sum(), 1)

# ── STOP WORDS ─────────────────────────────────────────────────────────────────
# Inglés de sklearn (el reasoning de Qwen, modalidad dominante, está en inglés)
# + prefijos que añade build_text para que no sean features espurias
STOP_WORDS = list(sklearn_text.ENGLISH_STOP_WORDS) + ["ocr", "trans", "reas"]

# ── VOCABULARIO Y MIN_DF DINÁMICOS ─────────────────────────────────────────────
print("\nCalculando vocabulario real del corpus...")

_counter = CountVectorizer(
    analyzer     = "word",
    ngram_range  = (1, 1),
    min_df       = 1,
    strip_accents= "unicode",
    stop_words   = STOP_WORDS,
)
_counter.fit(X_train + X_val)
VOCAB_SIZE = len(_counter.vocabulary_)

# min_df máximo: 0.5% del train, acotado entre 1 y 10
MIN_DF_MAX = max(1, min(10, int(len(X_train) * 0.005)))

# max_features: entre 10% del vocab y el vocab completo
MAX_FEATS_MIN = max(5_000, VOCAB_SIZE // 10)
MAX_FEATS_MAX = VOCAB_SIZE

print(f"Vocabulario real (unigramas, sin stop words): {VOCAB_SIZE:,} términos")
print(f"Rango max_features : [{MAX_FEATS_MIN:,} – {MAX_FEATS_MAX:,}]")
print(f"Rango min_df        : [1 – {MIN_DF_MAX}]")
print(f"Stop words          : {len(STOP_WORDS)} términos")

# ── ESPACIO DE BÚSQUEDA ────────────────────────────────────────────────────────
space = [
    # TF-IDF
    Integer(MAX_FEATS_MIN, MAX_FEATS_MAX,    name="max_features"),
    Integer(1,      2,                       name="ngram_min"),
    Integer(2,      4,                       name="ngram_max"),
    Integer(1,      MIN_DF_MAX,              name="min_df"),
    Categorical([True, False],               name="sublinear_tf"),
    Categorical(["word", "char_wb"],         name="analyzer"),

    # XGBoost
    Integer(100,  600,                       name="n_estimators"),
    Integer(3,    8,                         name="max_depth"),
    Real(0.01,    0.3,  prior="log-uniform", name="learning_rate"),
    Real(0.5,     1.0,                       name="subsample"),
    Real(0.5,     1.0,                       name="colsample_bytree"),
    Real(0.5,     1.0,                       name="colsample_bylevel"),
    Real(1,       10,   prior="log-uniform", name="min_child_weight"),
    Real(0,       5,                         name="gamma"),
    Real(0,       1,                         name="reg_alpha"),
    Real(0,       1,                         name="reg_lambda"),

    # Peso ensemble (XLM-R vs léxico)
    Real(0.3,     0.95,                      name="w_xlmr"),
]

# ── FUNCIÓN OBJETIVO ───────────────────────────────────────────────────────────
iteration = [0]

@use_named_args(space)
def objective(
    max_features, ngram_min, ngram_max, min_df, sublinear_tf, analyzer,
    n_estimators, max_depth, learning_rate, subsample, colsample_bytree,
    colsample_bylevel, min_child_weight, gamma, reg_alpha, reg_lambda,
    w_xlmr,
):
    iteration[0] += 1

    if ngram_min > ngram_max:
        return 0.0

    tfidf_kwargs = dict(
        max_features = int(max_features),
        ngram_range  = (int(ngram_min), int(ngram_max)),
        min_df       = int(min_df),
        sublinear_tf = bool(sublinear_tf),
        analyzer     = analyzer,
        strip_accents= "unicode",
        stop_words   = STOP_WORDS,
    )
    xgb_kwargs = dict(
        n_estimators      = int(n_estimators),
        max_depth         = int(max_depth),
        learning_rate     = float(learning_rate),
        subsample         = float(subsample),
        colsample_bytree  = float(colsample_bytree),
        colsample_bylevel = float(colsample_bylevel),
        min_child_weight  = float(min_child_weight),
        gamma             = float(gamma),
        reg_alpha         = float(reg_alpha),
        reg_lambda        = float(reg_lambda),
        scale_pos_weight  = scale_pos,
        eval_metric       = "logloss",
        random_state      = RANDOM_SEED,
        tree_method       = "hist",
    )

    # TF-IDF sobre todo el train para el CV
    vec = TfidfVectorizer(**tfidf_kwargs)
    X_tr_tfidf = vec.fit_transform(X_train)

    # CV paralelo sobre train
    cv = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)

    def _fold_score(train_idx, val_idx):
        clf_ = xgb.XGBClassifier(**{**xgb_kwargs, "n_jobs": 1})
        clf_.fit(X_tr_tfidf[train_idx], y_train[train_idx])
        probs = clf_.predict_proba(X_tr_tfidf[val_idx])[:, 1]
        preds = (probs >= 0.5).astype(int)
        return f1_score(y_train[val_idx], preds, average="macro", zero_division=0)

    fold_scores = Parallel(n_jobs=N_JOBS_CV, prefer="threads")(
        delayed(_fold_score)(tr, vl)
        for tr, vl in cv.split(X_tr_tfidf, y_train)
    )
    cv_f1 = float(np.mean(fold_scores))

    # Ensemble en val — métrica real que optimizamos
    vec_val = TfidfVectorizer(**tfidf_kwargs)
    X_tr2   = vec_val.fit_transform(X_train)
    X_vl2   = vec_val.transform(X_val)

    clf = xgb.XGBClassifier(**{**xgb_kwargs, "n_jobs": N_JOBS_XGB})
    clf.fit(X_tr2, y_train)
    lex_probs = clf.predict_proba(X_vl2)[:, 1]

    combined = np.where(
        (xlmr_probs_val >= 0.3) & (xlmr_probs_val <= 0.7),
        float(w_xlmr) * xlmr_probs_val + (1 - float(w_xlmr)) * lex_probs,
        xlmr_probs_val
    )
    ens_preds = (combined >= 0.5).astype(int)
    val_f1    = f1_score(y_val, ens_preds, average="macro", zero_division=0)

    print(f"  [{iteration[0]:>3}/{N_CALLS}] "
          f"CV-F1={cv_f1:.4f}  Val-F1={val_f1:.4f}  "
          f"w_xlmr={float(w_xlmr):.2f}  "
          f"ngram=({int(ngram_min)},{int(ngram_max)})  "
          f"feats={int(max_features)}  "
          f"depth={int(max_depth)}  lr={learning_rate:.3f}")

    return -val_f1

# ── BÚSQUEDA BAYESIANA ─────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"Iniciando búsqueda bayesiana ({N_CALLS} iter, {N_INITIAL} aleatorias)")
print(f"{'='*60}\n")

result = gp_minimize(
    func             = objective,
    dimensions       = space,
    n_calls          = N_CALLS,
    n_initial_points = N_INITIAL,
    acq_func         = "EI",
    random_state     = RANDOM_SEED,
    verbose          = False,
)

best_params = dict(zip([dim.name for dim in space], result.x))
best_val_f1 = -result.fun

print(f"\n{'='*60}")
print(f"MEJOR CONFIGURACIÓN  —  Val F1 macro = {best_val_f1:.4f}")
print(f"{'='*60}")
for k, v in best_params.items():
    print(f"  {k:25s}: {v}")

# ── REENTRENAR CON MEJORES PARÁMETROS ─────────────────────────────────────────
print("\nReentrenando modelo final con mejores parámetros...")

bp        = best_params
ngram_min = int(bp["ngram_min"])
ngram_max = int(bp["ngram_max"])
if ngram_min > ngram_max:
    ngram_max = ngram_min + 1

tfidf_final = TfidfVectorizer(
    max_features = int(bp["max_features"]),
    ngram_range  = (ngram_min, ngram_max),
    min_df       = int(bp["min_df"]),
    sublinear_tf = bool(bp["sublinear_tf"]),
    analyzer     = bp["analyzer"],
    strip_accents= "unicode",
    stop_words   = STOP_WORDS,
)
X_tr_final = tfidf_final.fit_transform(X_train)
X_vl_final = tfidf_final.transform(X_val)

clf_final = xgb.XGBClassifier(
    n_estimators      = int(bp["n_estimators"]),
    max_depth         = int(bp["max_depth"]),
    learning_rate     = float(bp["learning_rate"]),
    subsample         = float(bp["subsample"]),
    colsample_bytree  = float(bp["colsample_bytree"]),
    colsample_bylevel = float(bp["colsample_bylevel"]),
    min_child_weight  = float(bp["min_child_weight"]),
    gamma             = float(bp["gamma"]),
    reg_alpha         = float(bp["reg_alpha"]),
    reg_lambda        = float(bp["reg_lambda"]),
    scale_pos_weight  = scale_pos,
    eval_metric       = "logloss",
    random_state      = RANDOM_SEED,
    n_jobs            = N_JOBS_XGB,
    tree_method       = "hist",
)
clf_final.fit(X_tr_final, y_train)

lex_probs_val  = clf_final.predict_proba(X_vl_final)[:, 1]
w_xlmr         = float(bp["w_xlmr"])
combined_probs = w_xlmr * xlmr_probs_val + (1 - w_xlmr) * lex_probs_val
ensemble_preds = (combined_probs >= 0.5).astype(int)
lex_preds_only = (lex_probs_val >= 0.5).astype(int)
xlmr_preds_arr = np.array([p["pred"] for p in xlmr_preds])

# ── RESULTADOS FINALES ─────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("RESULTADOS FINALES EN VAL")
print(f"{'='*60}")

print(f"\n── XLM-R solo ───────────────────────────────────────")
print(classification_report(y_val, xlmr_preds_arr,
                             target_names=["No sarcasmo", "Sarcasmo"], digits=4))

print(f"\n── Léxico (TF-IDF + XGB) solo ───────────────────────")
print(classification_report(y_val, lex_preds_only,
                             target_names=["No sarcasmo", "Sarcasmo"], digits=4))

print(f"\n── Ensemble (w_xlmr={w_xlmr:.2f} / w_lex={1-w_xlmr:.2f}) ──")
print(classification_report(y_val, ensemble_preds,
                             target_names=["No sarcasmo", "Sarcasmo"], digits=4))

# ── ANÁLISIS EN FP/FN IDENTIFICADOS ───────────────────────────────────────────
FP_IDS = {
    "6881378887452970242", "6978595605941669125", "7276989936744615173",
    "6897721396332449026", "6961554475534257414", "6869807446953676034",
    "7075887138176339205", "6948114918243716357", "7087426367150640390",
    "7220824084593118469", "6774758485910441222", "6893664954109447425",
    "6912165587368119553", "6956643797392542981", "6933561551605566725",
    "7202520483760262406", "7059016269730680110", "6944860130466860293",
    "7184896691252874502", "7163463416307289349", "6942218282665610502",
    "7071293597764619562", "7061710476442406150", "7085105539394325766",
    "6814457272199105798",
}
FN_IDS = {
    "7061723251768036654", "7165922589175532805", "7072726688118967557",
    "7010191792410807558", "7087283296798870789", "7106154278691114245",
    "6991529983583866118", "6998563228590656773", "6911261711215430914",
    "6830438577638214918", "7141935831161359622", "7196476997793533190",
    "7138128003288075526", "7075653365119601926", "6802244300307434757",
    "6997040248878419205", "7063679805903932718", "6954208551694798082",
    "7217246198489484550", "7213218074936315141", "7014892556433542406",
    "7016428012904172806", "7020605795796520197",
}

rows = []
for i, (sample, xp) in enumerate(zip(val_data, xlmr_preds)):
    tid   = sample["id_Tiktok"]
    label = majority_label(sample["label"])
    rows.append({
        "id":        tid,
        "label":     label,
        "xlmr_prob": float(xp["prob"]),
        "xlmr_pred": int(xp["pred"]),
        "lex_prob":  float(lex_probs_val[i]),
        "lex_pred":  int(lex_preds_only[i]),
        "ens_prob":  float(combined_probs[i]),
        "ens_pred":  int(ensemble_preds[i]),
        "xlmr_ok":   int(xp["pred"]) == label,
        "lex_ok":    int(lex_preds_only[i]) == label,
        "ens_ok":    int(ensemble_preds[i]) == label,
        "is_fp":     tid in FP_IDS,
        "is_fn":     tid in FN_IDS,
    })

df = pd.DataFrame(rows)

for group, mask, label_str in [
    ("FP", df["is_fp"], "FALSOS POSITIVOS"),
    ("FN", df["is_fn"], "FALSOS NEGATIVOS"),
]:
    sub = df[mask]
    print(f"\n{'='*60}")
    print(f"IMPACTO EN {label_str} IDENTIFICADOS  [{len(sub)} casos]")
    print(f"{'='*60}")
    if sub.empty:
        print("  (ningún ID encontrado en val — revisar que los IDs coinciden con id_Tiktok)")
    else:
        print(sub[["id", "label", "xlmr_prob", "lex_prob", "ens_prob",
                   "xlmr_ok", "lex_ok", "ens_ok"]].to_string(index=False))
        print(f"\n  XLM-R correcto   : {sub['xlmr_ok'].sum()}/{len(sub)}")
        print(f"  Léxico correcto  : {sub['lex_ok'].sum()}/{len(sub)}")
        print(f"  Ensemble correcto: {sub['ens_ok'].sum()}/{len(sub)}")

# ── GUARDAR ────────────────────────────────────────────────────────────────────
output = {
    "model":            "tfidf-xgb-bayes + xlmr ensemble",
    "val_f1":           round(best_val_f1, 6),
    "weights":          {"xlmr": round(w_xlmr, 4), "lexical": round(1 - w_xlmr, 4)},
    "vocab_size":       VOCAB_SIZE,
    "stop_words_count": len(STOP_WORDS),
    "best_params":      best_params,
    "predictions": [
        {
            "id":        sample["id_Tiktok"],
            "label":     majority_label(sample["label"]),
            "pred":      int(ensemble_preds[i]),
            "prob":      round(float(combined_probs[i]), 6),
            "lex_prob":  round(float(lex_probs_val[i]), 6),
            "xlmr_prob": round(float(xlmr_probs_val[i]), 6),
        }
        for i, sample in enumerate(val_data)
    ],
}

with open(OUTPUT_PATH, "w") as f:
    json.dump(output, f, indent=2, ensure_ascii=False, cls=NpEncoder)

print(f"\n✓ Predicciones guardadas en {OUTPUT_PATH}")
print(f"✓ F1 macro ensemble final : {best_val_f1:.4f}")
print(f"  XLM-R solo              : {f1_score(y_val, xlmr_preds_arr, average='macro'):.4f}")

In [ ]:
"""
bayes_search.py
===============
Búsqueda bayesiana de hiperparámetros para TF-IDF + XGBoost.
Ensemble via meta-learner (LogisticRegression) entrenado con OOF predictions.

Cambios respecto a versión anterior:
  - Sin CV como métrica: train completo → val directo
  - Sin w_xlmr en el espacio de búsqueda
  - Meta-learner (LR sobre [lex_prob, xlmr_prob]) reemplaza el peso fijo
  - OOF del léxico sobre train para fitear el meta-learner sin leakage
  - Región de incertidumbre conservada: fuera de [0.3, 0.7] XLM-R decide solo

Requisitos:
    pip install scikit-optimize xgboost scikit-learn joblib pandas numpy
"""

import json
import warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.feature_extraction import text as sklearn_text
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold
from joblib import Parallel, delayed, cpu_count
from skopt import gp_minimize
from skopt.space import Real, Integer, Categorical
from skopt.utils import use_named_args
import xgboost as xgb

warnings.filterwarnings("ignore")

# ── RUTAS ──────────────────────────────────────────────────────────────────────
TRAIN_PATH  = "../data/last_task/train_new.json"
VAL_PATH    = "../data/last_task/val_new.json"
XLMR_PREDS  = "../data/last_task/xlm-roberta-base-79.6.json"
OUTPUT_PATH = "../data/last_task/lexical_preds_bayes.json"

N_CALLS      = 50
N_INITIAL    = 10
N_OOF_FOLDS  = 5   # solo para generar OOF probs, no como métrica de búsqueda
N_JOBS_OOF   = 5   # paralelismo al generar OOF
RANDOM_SEED  = 42

N_CORES = cpu_count()
print(f"CPUs disponibles: {N_CORES}")

# ── HELPERS ────────────────────────────────────────────────────────────────────
def majority_label(label_field) -> int:
    votes = label_field if isinstance(label_field, list) else [label_field]
    return int(sum(votes) > len(votes) / 2)

def build_text(sample: dict) -> str:
    ocr   = (sample.get("qwen_ocr")          or "").strip()
    trans = (sample.get("qwen_transcription") or "").strip()
    reas  = (sample.get("qwen_reasoning")     or "").strip()
    if trans == "<no-speech>":
        trans = ""
    parts = []
    if ocr:   parts.append("OCR: " + ocr)
    if trans: parts.append("TRANS: " + trans)
    if reas:  parts.append("REAS: " + reas)
    return " ".join(parts)

# ── JSON ENCODER para tipos numpy ──────────────────────────────────────────────
class NpEncoder(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, np.bool_):    return bool(o)
        if isinstance(o, np.integer):  return int(o)
        if isinstance(o, np.floating): return float(o)
        if isinstance(o, np.ndarray):  return o.tolist()
        return super().default(o)

# ── CARGA ──────────────────────────────────────────────────────────────────────
with open(TRAIN_PATH, "r") as f: train_data = json.load(f)
with open(VAL_PATH,   "r") as f: val_data   = json.load(f)
with open(XLMR_PREDS, "r") as f: xlmr_raw   = json.load(f)

xlmr_preds     = xlmr_raw["predictions"]
xlmr_probs_val = np.array([p["prob"] for p in xlmr_preds])

assert len(xlmr_preds) == len(val_data), "Mismatch XLM-R preds vs val_data"

X_train = [build_text(s) for s in train_data]
y_train = np.array([majority_label(s["label"]) for s in train_data])

X_val   = [build_text(s) for s in val_data]
y_val   = np.array([majority_label(s["label"]) for s in val_data])

print(f"Train: {len(X_train)} samples | Val: {len(X_val)} samples")
print(f"Train pos/neg: {y_train.sum()} / {(1-y_train).sum()}")

scale_pos = float((1 - y_train).sum()) / max(y_train.sum(), 1)

# ── XLM-R probs sobre train (necesarias para fitear el meta-learner) ───────────
# Asumimos que xlmr_raw contiene también train_predictions; si no, usamos
# un placeholder de 0.5 y el meta-learner aprenderá a ignorar esa señal.
xlmr_probs_train = np.array(
    [p["prob"] for p in xlmr_raw.get("train_predictions", [])]
)
if len(xlmr_probs_train) != len(train_data):
    print("⚠  No se encontraron train_predictions de XLM-R → usando 0.5 como placeholder")
    xlmr_probs_train = np.full(len(train_data), 0.5)

# ── STOP WORDS ─────────────────────────────────────────────────────────────────
STOP_WORDS = list(sklearn_text.ENGLISH_STOP_WORDS) + ["ocr", "trans", "reas"]

# ── VOCABULARIO Y MIN_DF DINÁMICOS ─────────────────────────────────────────────
print("\nCalculando vocabulario real del corpus...")

_counter = CountVectorizer(
    analyzer      = "word",
    ngram_range   = (1, 1),
    min_df        = 1,
    strip_accents = "unicode",
    stop_words    = STOP_WORDS,
)
_counter.fit(X_train + X_val)
VOCAB_SIZE = len(_counter.vocabulary_)

MIN_DF_MAX    = max(1, min(10, int(len(X_train) * 0.005)))
MAX_FEATS_MIN = max(5_000, VOCAB_SIZE // 10)
MAX_FEATS_MAX = VOCAB_SIZE

print(f"Vocabulario real (unigramas, sin stop words): {VOCAB_SIZE:,} términos")
print(f"Rango max_features : [{MAX_FEATS_MIN:,} – {MAX_FEATS_MAX:,}]")
print(f"Rango min_df        : [1 – {MIN_DF_MAX}]")
print(f"Stop words          : {len(STOP_WORDS)} términos")

# ── ESPACIO DE BÚSQUEDA ────────────────────────────────────────────────────────
# w_xlmr eliminado: el ensemble es ahora un meta-learner aprendido
space = [
    # TF-IDF
    Integer(MAX_FEATS_MIN, MAX_FEATS_MAX,    name="max_features"),
    Integer(1,      2,                       name="ngram_min"),
    Integer(2,      4,                       name="ngram_max"),
    Integer(1,      MIN_DF_MAX,              name="min_df"),
    Categorical([True, False],               name="sublinear_tf"),
    Categorical(["word", "char_wb"],         name="analyzer"),

    # XGBoost
    Integer(100,  600,                       name="n_estimators"),
    Integer(3,    8,                         name="max_depth"),
    Real(0.01,    0.3,  prior="log-uniform", name="learning_rate"),
    Real(0.5,     1.0,                       name="subsample"),
    Real(0.5,     1.0,                       name="colsample_bytree"),
    Real(0.5,     1.0,                       name="colsample_bylevel"),
    Real(1,       10,   prior="log-uniform", name="min_child_weight"),
    Real(0,       5,                         name="gamma"),
    Real(0,       1,                         name="reg_alpha"),
    Real(0,       1,                         name="reg_lambda"),
]

# ── FUNCIÓN: genera OOF lex_probs sobre train ─────────────────────────────────
def get_oof_lex_probs(X_tr_tfidf, xgb_kwargs) -> np.ndarray:
    """
    Genera out-of-fold predictions del léxico sobre train.
    Se usan para fitear el meta-learner sin data leakage.
    """
    oof_probs = np.zeros(len(y_train))
    cv        = StratifiedKFold(n_splits=N_OOF_FOLDS, shuffle=True,
                                random_state=RANDOM_SEED)

    def _fold(train_idx, val_idx):
        clf_ = xgb.XGBClassifier(**{**xgb_kwargs, "n_jobs": 1})
        clf_.fit(X_tr_tfidf[train_idx], y_train[train_idx])
        return val_idx, clf_.predict_proba(X_tr_tfidf[val_idx])[:, 1]

    results = Parallel(n_jobs=N_JOBS_OOF, prefer="threads")(
        delayed(_fold)(tr, vl)
        for tr, vl in cv.split(X_tr_tfidf, y_train)
    )
    for val_idx, probs in results:
        oof_probs[val_idx] = probs

    return oof_probs

# ── FUNCIÓN: ensemble con meta-learner + región de incertidumbre ───────────────
def ensemble_predict(lex_probs_v, xlmr_probs_v, meta_lr):
    """
    - Región segura (xlmr_prob fuera de [0.3, 0.7]): XLM-R decide solo.
    - Región incierta (xlmr_prob ∈ [0.3, 0.7]): meta-learner decide.
    Devuelve probabilidades finales y predicciones binarias.
    """
    uncertain = (xlmr_probs_v >= 0.3) & (xlmr_probs_v <= 0.7)
    final_probs = xlmr_probs_v.copy()

    if uncertain.sum() > 0:
        X_meta = np.column_stack([lex_probs_v[uncertain],
                                  xlmr_probs_v[uncertain]])
        final_probs[uncertain] = meta_lr.predict_proba(X_meta)[:, 1]

    return final_probs, (final_probs >= 0.5).astype(int)

# ── FUNCIÓN OBJETIVO ───────────────────────────────────────────────────────────
iteration = [0]

@use_named_args(space)
def objective(
    max_features, ngram_min, ngram_max, min_df, sublinear_tf, analyzer,
    n_estimators, max_depth, learning_rate, subsample, colsample_bytree,
    colsample_bylevel, min_child_weight, gamma, reg_alpha, reg_lambda,
):
    iteration[0] += 1

    if ngram_min > ngram_max:
        return 0.0

    tfidf_kwargs = dict(
        max_features  = int(max_features),
        ngram_range   = (int(ngram_min), int(ngram_max)),
        min_df        = int(min_df),
        sublinear_tf  = bool(sublinear_tf),
        analyzer      = analyzer,
        strip_accents = "unicode",
        stop_words    = STOP_WORDS,
    )
    xgb_kwargs = dict(
        n_estimators      = int(n_estimators),
        max_depth         = int(max_depth),
        learning_rate     = float(learning_rate),
        subsample         = float(subsample),
        colsample_bytree  = float(colsample_bytree),
        colsample_bylevel = float(colsample_bylevel),
        min_child_weight  = float(min_child_weight),
        gamma             = float(gamma),
        reg_alpha         = float(reg_alpha),
        reg_lambda        = float(reg_lambda),
        scale_pos_weight  = scale_pos,
        eval_metric       = "logloss",
        random_state      = RANDOM_SEED,
        tree_method       = "hist",
        n_jobs            = 1,
    )

    # ── 1. TF-IDF ──────────────────────────────────────────────────────────────
    vec         = TfidfVectorizer(**tfidf_kwargs)
    X_tr_tfidf  = vec.fit_transform(X_train)
    X_vl_tfidf  = vec.transform(X_val)

    # ── 2. XGBoost entrenado en train completo → probs en val ──────────────────
    clf = xgb.XGBClassifier(**xgb_kwargs)
    clf.fit(X_tr_tfidf, y_train)
    lex_probs_val = clf.predict_proba(X_vl_tfidf)[:, 1]

    # ── 3. OOF lex_probs en train → fitear meta-learner ────────────────────────
    oof_lex_probs = get_oof_lex_probs(X_tr_tfidf, xgb_kwargs)

    # Meta-learner: LR sobre [lex_prob, xlmr_prob] del train
    # Solo muestras de la región incierta (consistente con inferencia)
    uncertain_train = (
        (xlmr_probs_train >= 0.3) & (xlmr_probs_train <= 0.7)
    )
    if uncertain_train.sum() < 10:
        # Si hay muy pocas muestras inciertas, entrenar con todas
        uncertain_train = np.ones(len(y_train), dtype=bool)

    X_meta_train = np.column_stack([
        oof_lex_probs[uncertain_train],
        xlmr_probs_train[uncertain_train],
    ])
    y_meta_train = y_train[uncertain_train]

    meta_lr = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
    meta_lr.fit(X_meta_train, y_meta_train)

    # ── 4. Ensemble en val ─────────────────────────────────────────────────────
    _, ens_preds = ensemble_predict(lex_probs_val, xlmr_probs_val, meta_lr)
    val_f1       = f1_score(y_val, ens_preds, average="macro", zero_division=0)

    print(f"  [{iteration[0]:>3}/{N_CALLS}] "
          f"Val-F1={val_f1:.4f}  "
          f"ngram=({int(ngram_min)},{int(ngram_max)})  "
          f"feats={int(max_features)}  "
          f"depth={int(max_depth)}  lr={learning_rate:.3f}  "
          f"uncertain_train={uncertain_train.sum()}")

    return -val_f1

# ── BÚSQUEDA BAYESIANA ─────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"Iniciando búsqueda bayesiana ({N_CALLS} iter, {N_INITIAL} aleatorias)")
print(f"{'='*60}\n")

result = gp_minimize(
    func             = objective,
    dimensions       = space,
    n_calls          = N_CALLS,
    n_initial_points = N_INITIAL,
    acq_func         = "EI",
    random_state     = RANDOM_SEED,
    verbose          = False,
)

best_params = dict(zip([dim.name for dim in space], result.x))
best_val_f1 = -result.fun

print(f"\n{'='*60}")
print(f"MEJOR CONFIGURACIÓN  —  Val F1 macro = {best_val_f1:.4f}")
print(f"{'='*60}")
for k, v in best_params.items():
    print(f"  {k:25s}: {v}")

# ── REENTRENAR CON MEJORES PARÁMETROS ─────────────────────────────────────────
print("\nReentrenando modelo final con mejores parámetros...")

bp        = best_params
ngram_min = int(bp["ngram_min"])
ngram_max = int(bp["ngram_max"])
if ngram_min > ngram_max:
    ngram_max = ngram_min + 1

tfidf_final = TfidfVectorizer(
    max_features  = int(bp["max_features"]),
    ngram_range   = (ngram_min, ngram_max),
    min_df        = int(bp["min_df"]),
    sublinear_tf  = bool(bp["sublinear_tf"]),
    analyzer      = bp["analyzer"],
    strip_accents = "unicode",
    stop_words    = STOP_WORDS,
)
X_tr_final = tfidf_final.fit_transform(X_train)
X_vl_final = tfidf_final.transform(X_val)

xgb_final_kwargs = dict(
    n_estimators      = int(bp["n_estimators"]),
    max_depth         = int(bp["max_depth"]),
    learning_rate     = float(bp["learning_rate"]),
    subsample         = float(bp["subsample"]),
    colsample_bytree  = float(bp["colsample_bytree"]),
    colsample_bylevel = float(bp["colsample_bylevel"]),
    min_child_weight  = float(bp["min_child_weight"]),
    gamma             = float(bp["gamma"]),
    reg_alpha         = float(bp["reg_alpha"]),
    reg_lambda        = float(bp["reg_lambda"]),
    scale_pos_weight  = scale_pos,
    eval_metric       = "logloss",
    random_state      = RANDOM_SEED,
    tree_method       = "hist",
    n_jobs            = -1,   # todos los cores en el modelo final
)

clf_final = xgb.XGBClassifier(**xgb_final_kwargs)
clf_final.fit(X_tr_final, y_train)
lex_probs_val  = clf_final.predict_proba(X_vl_final)[:, 1]

# Meta-learner final: OOF sobre train completo con mejores params
oof_lex_final = get_oof_lex_probs(X_tr_final, {**xgb_final_kwargs, "n_jobs": 1})

uncertain_train = (xlmr_probs_train >= 0.3) & (xlmr_probs_train <= 0.7)
if uncertain_train.sum() < 10:
    uncertain_train = np.ones(len(y_train), dtype=bool)

meta_lr_final = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
meta_lr_final.fit(
    np.column_stack([oof_lex_final[uncertain_train],
                     xlmr_probs_train[uncertain_train]]),
    y_train[uncertain_train],
)

print(f"Meta-learner coefs: lex={meta_lr_final.coef_[0][0]:.3f}  "
      f"xlmr={meta_lr_final.coef_[0][1]:.3f}  "
      f"bias={meta_lr_final.intercept_[0]:.3f}")

combined_probs, ensemble_preds = ensemble_predict(
    lex_probs_val, xlmr_probs_val, meta_lr_final
)

lex_preds_only = (lex_probs_val >= 0.5).astype(int)
xlmr_preds_arr = np.array([p["pred"] for p in xlmr_preds])

# ── RESULTADOS FINALES ─────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("RESULTADOS FINALES EN VAL")
print(f"{'='*60}")

print(f"\n── XLM-R solo ───────────────────────────────────────")
print(classification_report(y_val, xlmr_preds_arr,
                             target_names=["No sarcasmo", "Sarcasmo"], digits=4))

print(f"\n── Léxico (TF-IDF + XGB) solo ───────────────────────")
print(classification_report(y_val, lex_preds_only,
                             target_names=["No sarcasmo", "Sarcasmo"], digits=4))

print(f"\n── Ensemble (meta-learner LR) ────────────────────────")
print(classification_report(y_val, ensemble_preds,
                             target_names=["No sarcasmo", "Sarcasmo"], digits=4))

# ── ANÁLISIS EN FP/FN IDENTIFICADOS ───────────────────────────────────────────
FP_IDS = {
    "6881378887452970242", "6978595605941669125", "7276989936744615173",
    "6897721396332449026", "6961554475534257414", "6869807446953676034",
    "7075887138176339205", "6948114918243716357", "7087426367150640390",
    "7220824084593118469", "6774758485910441222", "6893664954109447425",
    "6912165587368119553", "6956643797392542981", "6933561551605566725",
    "7202520483760262406", "7059016269730680110", "6944860130466860293",
    "7184896691252874502", "7163463416307289349", "6942218282665610502",
    "7071293597764619562", "7061710476442406150", "7085105539394325766",
    "6814457272199105798",
}
FN_IDS = {
    "7061723251768036654", "7165922589175532805", "7072726688118967557",
    "7010191792410807558", "7087283296798870789", "7106154278691114245",
    "6991529983583866118", "6998563228590656773", "6911261711215430914",
    "6830438577638214918", "7141935831161359622", "7196476997793533190",
    "7138128003288075526", "7075653365119601926", "6802244300307434757",
    "6997040248878419205", "7063679805903932718", "6954208551694798082",
    "7217246198489484550", "7213218074936315141", "7014892556433542406",
    "7016428012904172806", "7020605795796520197",
}

rows = []
for i, (sample, xp) in enumerate(zip(val_data, xlmr_preds)):
    tid   = sample["id_Tiktok"]
    label = majority_label(sample["label"])
    rows.append({
        "id":        tid,
        "label":     label,
        "xlmr_prob": float(xp["prob"]),
        "xlmr_pred": int(xp["pred"]),
        "lex_prob":  float(lex_probs_val[i]),
        "lex_pred":  int(lex_preds_only[i]),
        "ens_prob":  float(combined_probs[i]),
        "ens_pred":  int(ensemble_preds[i]),
        "xlmr_ok":   int(xp["pred"]) == label,
        "lex_ok":    int(lex_preds_only[i]) == label,
        "ens_ok":    int(ensemble_preds[i]) == label,
        "is_fp":     tid in FP_IDS,
        "is_fn":     tid in FN_IDS,
    })

df = pd.DataFrame(rows)

for group, mask, label_str in [
    ("FP", df["is_fp"], "FALSOS POSITIVOS"),
    ("FN", df["is_fn"], "FALSOS NEGATIVOS"),
]:
    sub = df[mask]
    print(f"\n{'='*60}")
    print(f"IMPACTO EN {label_str} IDENTIFICADOS  [{len(sub)} casos]")
    print(f"{'='*60}")
    if sub.empty:
        print("  (ningún ID encontrado en val)")
    else:
        print(sub[["id", "label", "xlmr_prob", "lex_prob", "ens_prob",
                   "xlmr_ok", "lex_ok", "ens_ok"]].to_string(index=False))
        print(f"\n  XLM-R correcto   : {sub['xlmr_ok'].sum()}/{len(sub)}")
        print(f"  Léxico correcto  : {sub['lex_ok'].sum()}/{len(sub)}")
        print(f"  Ensemble correcto: {sub['ens_ok'].sum()}/{len(sub)}")

# ── GUARDAR ────────────────────────────────────────────────────────────────────
output = {
    "model":             "tfidf-xgb-bayes + xlmr meta-learner ensemble",
    "val_f1":            round(best_val_f1, 6),
    "meta_learner":      {
        "coef_lex":  round(float(meta_lr_final.coef_[0][0]), 4),
        "coef_xlmr": round(float(meta_lr_final.coef_[0][1]), 4),
        "intercept": round(float(meta_lr_final.intercept_[0]), 4),
    },
    "vocab_size":        VOCAB_SIZE,
    "stop_words_count":  len(STOP_WORDS),
    "best_params":       best_params,
    "predictions": [
        {
            "id":        sample["id_Tiktok"],
            "label":     majority_label(sample["label"]),
            "pred":      int(ensemble_preds[i]),
            "prob":      round(float(combined_probs[i]), 6),
            "lex_prob":  round(float(lex_probs_val[i]), 6),
            "xlmr_prob": round(float(xlmr_probs_val[i]), 6),
        }
        for i, sample in enumerate(val_data)
    ],
}

with open(OUTPUT_PATH, "w") as f:
    json.dump(output, f, indent=2, ensure_ascii=False, cls=NpEncoder)

print(f"\n✓ Predicciones guardadas en {OUTPUT_PATH}")
print(f"✓ F1 macro ensemble final : {best_val_f1:.4f}")
print(f"  XLM-R solo              : {f1_score(y_val, xlmr_preds_arr, average='macro'):.4f}")

In [ ]:
import json
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
from scipy.stats import entropy

# ========= CONFIG =========
train_path = "../data/last_task/train.json"
val_path = "../data/last_task/val.json"
key = "task2"

# ========= LOAD =========
# ========= LOAD =========
def load_labels(path):
    with open(path, "r") as f:
        data = json.load(f)
    return [item[key] for item in data]

train_labels = load_labels(train_path)
val_labels = load_labels(val_path)

# ========= CLEAN =========
def clean(labels, expected_len=3):
    labels = [x for x in labels if len(x) == expected_len]
    return labels

train_labels = clean(train_labels)
val_labels = clean(val_labels)

# ========= TRANSFORM =========
def to_2class(labels):
    # ejemplo: clase 0 vs (1+2)
    return [[x[0], x[1] + x[2]] for x in labels]

# ========= ANALYSIS =========
def analyze(labels, name):
    print(f"\n===== {name} =====")
    
    counter = Counter([tuple(x) for x in labels])
    
    print("\nTop combinaciones:")
    for combo, count in counter.most_common(10):
        print(f"{combo}: {count}")
    
    arr = np.array(labels)
    
    class_sums = arr.sum(axis=0)
    print("\nSuma por clase:", class_sums)
    
    norm = arr / (arr.sum(axis=1, keepdims=True) + 1e-8)
    
    print("Media:", norm.mean(axis=0))
    print("Confianza media:", norm.max(axis=1).mean())
    print("Entropía media:", np.mean([entropy(x) for x in norm]))
    
    return arr, norm

# ========= RUN =========

# --- 3 CLASES ---
train_arr3, norm_train3 = analyze(train_labels, "TRAIN (3 clases)")
val_arr3, norm_val3 = analyze(val_labels, "VAL (3 clases)")

# --- 2 CLASES ---
train_labels2 = to_2class(train_labels)
val_labels2 = to_2class(val_labels)

train_arr2, norm_train2 = analyze(train_labels2, "TRAIN (2 clases)")
val_arr2, norm_val2 = analyze(val_labels2, "VAL (2 clases)")

# ========= PLOTS =========
plt.figure(figsize=(12,4))

# Confianza
plt.subplot(1,2,1)
plt.hist(norm_train3.max(axis=1), bins=50, alpha=0.5, label="train 3c")
plt.hist(norm_val3.max(axis=1), bins=50, alpha=0.5, label="val 3c")
plt.hist(norm_train2.max(axis=1), bins=50, alpha=0.5, label="train 2c")
plt.hist(norm_val2.max(axis=1), bins=50, alpha=0.5, label="val 2c")
plt.title("Confianza")
plt.legend()

# Entropía
plt.subplot(1,2,2)
plt.hist([entropy(x) for x in norm_train3], bins=50, alpha=0.5, label="train 3c")
plt.hist([entropy(x) for x in norm_train2], bins=50, alpha=0.5, label="train 2c")
plt.title("Entropía")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
import json
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
from scipy.stats import entropy

# ========= CONFIG =========
train_path = "../data/last_task/train_3.json"
val_path = "../data/last_task/val_3.json"
key = "task3"

# ========= LOAD =========
# ========= LOAD =========
def load_labels(path):
    with open(path, "r") as f:
        data = json.load(f)
    return [item[key] for item in data]

def to_multihot(labels, num_classes=None):
    """
    Convierte labels a formato multi-hot [N, C]
    
    Soporta:
    - lista de índices: [ [0,2], [1], ... ]
    - ya binario: [ [1,0,1], ... ]
    """
    # Detectar si ya es multi-hot
    if isinstance(labels[0][0], (int, float)) and max(labels[0]) <= 1:
        return np.array(labels)

    # Si son índices
    if num_classes is None:
        num_classes = max(max(x) for x in labels) + 1

    multihot = np.zeros((len(labels), num_classes), dtype=int)

    for i, sample in enumerate(labels):
        for cls in sample:
            multihot[i, cls] = 1

    return multihot

def analyze_multilabel(arr, name):
    print(f"\n===== {name} =====")

    N, C = arr.shape

    # 1. Frecuencia absoluta
    class_counts = arr.sum(axis=0)
    print("\nPositivos por clase:", class_counts)

    # 2. Frecuencia relativa
    class_freq = class_counts / N
    print("Frecuencia por clase:", class_freq)

    # 🔥 3. Media de frecuencia por etiqueta
    mean_freq = class_freq.mean()
    print("\nMedia de frecuencia por etiqueta:", mean_freq)

    # 🔥 4. Media de etiquetas activas por sample (cardinalidad)
    avg_labels_per_sample = arr.sum(axis=1).mean()
    print("Media labels activas por sample:", avg_labels_per_sample)

    # 5. Distribución cardinalidad
    print("\nDistribución (#labels activas):")
    print(Counter(arr.sum(axis=1)))

    # 6. Top combinaciones
    combos = Counter([tuple(x) for x in arr])
    print("\nTop combinaciones:")
    for combo, count in combos.most_common(10):
        print(combo, count)

    # 7. Checks críticos
    print("\nClases sin positivos:", np.where(class_counts == 0)[0])
    print("Clases siempre activas:", np.where(class_counts == N)[0])

    return arr

def plot_multilabel(arr, title=""):
    plt.figure(figsize=(12,4))

    # Labels activas por sample
    plt.subplot(1,2,1)
    plt.hist(arr.sum(axis=1), bins=range(1, arr.shape[1]+2))
    plt.title(f"{title} - Labels activas")

    # Frecuencia por clase
    plt.subplot(1,2,2)
    plt.bar(range(arr.shape[1]), arr.sum(axis=0))
    plt.title(f"{title} - Frecuencia por clase")

    plt.tight_layout()
    plt.show()

# ========= LOAD =========
train_labels = load_labels(train_path)
val_labels   = load_labels(val_path)

# ========= CLEAN =========
def clean(labels):
    return [x for x in labels if len(x) > 0]

train_labels = clean(train_labels)
val_labels   = clean(val_labels)

# ========= TO MULTIHOT =========
train_arr = to_multihot(train_labels)
val_arr   = to_multihot(val_labels, num_classes=train_arr.shape[1])

# ========= ANALYZE =========
train_arr = analyze_multilabel(train_arr, "TRAIN")
val_arr   = analyze_multilabel(val_arr, "VAL")

# ========= PLOTS =========
plot_multilabel(train_arr, "TRAIN")
plot_multilabel(val_arr, "VAL")